# 03b — Medication Feature Engineering

This notebook extracts **9 medication features** from `prescriptions.csv` and merges them into the existing `model_features` table in DuckDB.

**Run order:** After `03_Feature_Engineering.ipynb`, before `04_Data_Preprocessing.ipynb`

### Features Added
| Feature | Type | Description |
|---|---|---|
| `on_loop_diuretic` | binary | Furosemide / bumetanide / torsemide prescribed |
| `on_ace_arb` | binary | ACE inhibitor or ARB prescribed |
| `on_beta_blocker` | binary | Beta-blocker prescribed |
| `on_aldosterone_antagonist` | binary | Spironolactone or eplerenone prescribed |
| `on_digoxin` | binary | Digoxin prescribed |
| `on_anticoagulant` | binary | Warfarin / heparin / NOAC prescribed |
| `num_unique_drugs` | numeric | Total unique drugs per admission (polypharmacy proxy) |
| `furosemide_max_dose` | numeric | Max single furosemide dose (mg, IV-equivalent) |
| `gdmt_score` | numeric | Count of guideline-directed therapy classes present (0–4) |

In [ ]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
import duckdb
from utils import get_db_connection

DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'dataset')
con = get_db_connection()

print('Connected to DuckDB')
print('Existing tables:', con.execute('SHOW TABLES').fetchdf()['name'].tolist())

## Step 1 — Load HF Cohort IDs
Only extract features for the 4,508 admissions in our labeled cohort.

In [ ]:

# Try loading from DuckDB first (if notebooks 01-03 have been run)
# Fall back to building cohort from raw CSVs if DuckDB is empty

tables = con.execute('SHOW TABLES').fetchdf()['name'].tolist()

if 'hf_labeled' in tables:
    cohort = con.execute('SELECT hadm_id, subject_id FROM hf_labeled').fetchdf()
    hf_hadm_ids = set(cohort['hadm_id'].tolist())
    print(f'Loaded cohort from DuckDB hf_labeled: {len(hf_hadm_ids):,} admissions')

else:
    print('hf_labeled not found in DuckDB — building cohort from raw CSVs...')

    admissions = pd.read_csv(
        os.path.join(DATA_PATH, 'admissions.csv'),
        usecols=['hadm_id', 'subject_id', 'admittime', 'dischtime']
    )
    patients = pd.read_csv(
        os.path.join(DATA_PATH, 'patients.csv'),
        usecols=['subject_id', 'dod']
    )
    hf_diag = pd.read_csv(
        os.path.join(DATA_PATH, 'heart_diagnoses.csv'),
        usecols=['hadm_id']
    )

    # Merge to get HF admissions with patient info
    hf_adm = admissions[admissions['hadm_id'].isin(hf_diag['hadm_id'])].copy()
    hf_adm = hf_adm.merge(patients, on='subject_id', how='left')

    hf_adm['admittime'] = pd.to_datetime(hf_adm['admittime'])
    hf_adm['dischtime'] = pd.to_datetime(hf_adm['dischtime'])
    hf_adm['dod']       = pd.to_datetime(hf_adm['dod'], errors='coerce')

    # Exclude patients who died within 30 days of discharge (same logic as Notebook 02)
    death_within_30 = (
        hf_adm['dod'].notna() &
        ((hf_adm['dod'] - hf_adm['dischtime']).dt.days <= 30)
    )
    hf_adm = hf_adm[~death_within_30].copy()

    hf_hadm_ids = set(hf_adm['hadm_id'].tolist())
    print(f'Built cohort from CSVs: {len(hf_hadm_ids):,} admissions (deaths within 30d excluded)')


## Step 2 — Load & Filter Prescriptions
The full `prescriptions.csv` is 3.2GB with 20M rows. We load only the columns we need and immediately filter to HF patients.

In [ ]:
RX_PATH = os.path.join(DATA_PATH, 'prescriptions.csv')

print('Loading prescriptions.csv (this may take ~1-2 min)...')
rx = pd.read_csv(
    RX_PATH,
    usecols=['hadm_id', 'drug', 'dose_val_rx', 'dose_unit_rx', 'route'],
    dtype={'hadm_id': 'Int64', 'drug': 'str',
           'dose_val_rx': 'str', 'dose_unit_rx': 'str', 'route': 'str'},
    low_memory=False
)

# Filter to HF cohort only
rx = rx[rx['hadm_id'].isin(hf_hadm_ids)].copy()
rx['drug_lower'] = rx['drug'].str.lower().fillna('')

print(f'Total prescription rows for HF cohort: {len(rx):,}')
print(f'Admissions with any prescription:       {rx["hadm_id"].nunique():,} / {len(hf_hadm_ids):,}')

## Step 3 — Define Drug Keyword Lists

In [ ]:
LOOP_DIURETICS = ['furosemide', 'lasix', 'bumetanide', 'bumex', 'torsemide', 'demadex']

ACE_ARB = [
    # ACE inhibitors
    'lisinopril', 'enalapril', 'captopril', 'ramipril',
    'perindopril', 'quinapril', 'benazepril', 'fosinopril',
    # ARBs
    'losartan', 'valsartan', 'irbesartan', 'olmesartan',
    'candesartan', 'telmisartan', 'azilsartan'
]

BETA_BLOCKERS = [
    'carvedilol', 'metoprolol', 'bisoprolol', 'atenolol',
    'propranolol', 'nebivolol', 'labetalol'
]

ALDOSTERONE_ANTAGONISTS = ['spironolactone', 'aldactone', 'eplerenone', 'inspra']

DIGOXIN = ['digoxin', 'lanoxin']

ANTICOAGULANTS = [
    'warfarin', 'coumadin', 'heparin', 'enoxaparin', 'lovenox',
    'apixaban', 'eliquis', 'rivaroxaban', 'xarelto',
    'dabigatran', 'pradaxa', 'fondaparinux'
]

def matches_any(drug_lower, keyword_list):
    return any(kw in drug_lower for kw in keyword_list)

print('Drug keyword lists defined.')
print(f'  Loop diuretics:          {len(LOOP_DIURETICS)} keywords')
print(f'  ACE inhibitors / ARBs:   {len(ACE_ARB)} keywords')
print(f'  Beta-blockers:           {len(BETA_BLOCKERS)} keywords')
print(f'  Aldosterone antagonists: {len(ALDOSTERONE_ANTAGONISTS)} keywords')
print(f'  Digoxin:                 {len(DIGOXIN)} keywords')
print(f'  Anticoagulants:          {len(ANTICOAGULANTS)} keywords')

## Step 4 — Extract Furosemide Max Dose
Convert oral furosemide to IV-equivalent (oral bioavailability ~50% in HF patients).

In [ ]:
# Filter to furosemide rows
furo_mask = rx['drug_lower'].apply(lambda x: matches_any(x, ['furosemide', 'lasix']))
furo_rx = rx[furo_mask].copy()

# Extract numeric dose (handles strings like '40', '20-40' → takes first number)
furo_rx['dose_num'] = (
    furo_rx['dose_val_rx']
    .str.extract(r'([\d\.]+)')[0]
    .astype(float)
)

# Convert oral to IV-equivalent dose
def oral_to_iv(row):
    dose = row['dose_num']
    if pd.isna(dose):
        return np.nan
    route = str(row['route']).upper()
    if any(r in route for r in ['IV', 'INTRAVENOUS', 'IM']):
        return dose          # IV: use as-is
    else:
        return dose * 0.5    # Oral: 50% bioavailability adjustment

furo_rx['iv_equiv_dose'] = furo_rx.apply(oral_to_iv, axis=1)

# Max dose per admission
furo_max = (
    furo_rx.groupby('hadm_id')['iv_equiv_dose']
    .max()
    .reset_index()
    .rename(columns={'iv_equiv_dose': 'furosemide_max_dose'})
)

print(f'Furosemide prescribed in {len(furo_max):,} admissions ({len(furo_max)/len(hf_hadm_ids)*100:.1f}%)')
print(f'Median max IV-equiv dose: {furo_max["furosemide_max_dose"].median():.0f} mg')
print(f'Max IV-equiv dose seen:   {furo_max["furosemide_max_dose"].max():.0f} mg')
print()
print('Dose distribution (IV-equiv mg):')
print(furo_max['furosemide_max_dose'].describe().round(1))

## Step 5 — Build Medication Feature Table

In [ ]:
print('Building medication features per admission...')

med_rows = []
for hadm_id, group in rx.groupby('hadm_id'):
    drugs = group['drug_lower']

    on_loop    = int(drugs.apply(lambda x: matches_any(x, LOOP_DIURETICS)).any())
    on_ace_arb = int(drugs.apply(lambda x: matches_any(x, ACE_ARB)).any())
    on_bb      = int(drugs.apply(lambda x: matches_any(x, BETA_BLOCKERS)).any())
    on_aldo    = int(drugs.apply(lambda x: matches_any(x, ALDOSTERONE_ANTAGONISTS)).any())
    on_dig     = int(drugs.apply(lambda x: matches_any(x, DIGOXIN)).any())
    on_anticoag= int(drugs.apply(lambda x: matches_any(x, ANTICOAGULANTS)).any())
    n_unique   = drugs.nunique()

    # GDMT score: how many of the 4 guideline therapy classes are present
    gdmt = on_loop + on_ace_arb + on_bb + on_aldo

    med_rows.append({
        'hadm_id':                   hadm_id,
        'on_loop_diuretic':          on_loop,
        'on_ace_arb':                on_ace_arb,
        'on_beta_blocker':           on_bb,
        'on_aldosterone_antagonist': on_aldo,
        'on_digoxin':                on_dig,
        'on_anticoagulant':          on_anticoag,
        'num_unique_drugs':          n_unique,
        'gdmt_score':                gdmt,
    })

med_features = pd.DataFrame(med_rows)

# Merge furosemide max dose
med_features = med_features.merge(furo_max, on='hadm_id', how='left')
med_features['furosemide_max_dose'] = med_features['furosemide_max_dose'].fillna(0)

print(f'medication_features shape: {med_features.shape}')
print()
print('Drug class prevalence across HF cohort:')
binary_cols = ['on_loop_diuretic','on_ace_arb','on_beta_blocker',
               'on_aldosterone_antagonist','on_digoxin','on_anticoagulant']
for col in binary_cols:
    n   = med_features[col].sum()
    pct = med_features[col].mean() * 100
    print(f'  {col:35s}: {n:,} ({pct:.1f}%)')

print()
print(f'  num_unique_drugs  — mean: {med_features["num_unique_drugs"].mean():.1f}, median: {med_features["num_unique_drugs"].median():.0f}')
print(f'  furosemide_dose   — mean: {med_features["furosemide_max_dose"].mean():.1f} mg')
print(f'  gdmt_score        — mean: {med_features["gdmt_score"].mean():.2f} / 4')

## Step 6 — Save to DuckDB

In [ ]:
con.execute('DROP TABLE IF EXISTS medication_features')
con.execute('CREATE TABLE medication_features AS SELECT * FROM med_features')

n = con.execute('SELECT COUNT(*) FROM medication_features').fetchone()[0]
print(f'Saved medication_features to DuckDB: {n:,} rows')
print()
print(con.execute('DESCRIBE medication_features').fetchdf().to_string(index=False))

## Step 7 — Merge Into `model_features`
Join medication features into the existing `model_features` table built by Notebook 03.

In [ ]:
# Verify model_features exists
tables = con.execute('SHOW TABLES').fetchdf()['name'].tolist()
assert 'model_features' in tables, 'ERROR: model_features table not found — run 03_Feature_Engineering.ipynb first'

before_cols = len(con.execute('DESCRIBE model_features').fetchdf())
print(f'model_features before merge: {before_cols} columns')

con.execute("""
CREATE OR REPLACE TABLE model_features AS
SELECT
    mf.*,
    COALESCE(med.on_loop_diuretic,          0) AS on_loop_diuretic,
    COALESCE(med.on_ace_arb,                0) AS on_ace_arb,
    COALESCE(med.on_beta_blocker,           0) AS on_beta_blocker,
    COALESCE(med.on_aldosterone_antagonist, 0) AS on_aldosterone_antagonist,
    COALESCE(med.on_digoxin,               0) AS on_digoxin,
    COALESCE(med.on_anticoagulant,         0) AS on_anticoagulant,
    COALESCE(med.num_unique_drugs,         0) AS num_unique_drugs,
    COALESCE(med.furosemide_max_dose,      0) AS furosemide_max_dose,
    COALESCE(med.gdmt_score,               0) AS gdmt_score
FROM model_features mf
LEFT JOIN medication_features med ON mf.hadm_id = med.hadm_id
""")

after_cols = len(con.execute('DESCRIBE model_features').fetchdf())
rows       = con.execute('SELECT COUNT(*) FROM model_features').fetchone()[0]

print(f'model_features after merge:  {after_cols} columns')
print(f'New columns added:           {after_cols - before_cols}')
print(f'Total rows:                  {rows:,}')

## Step 8 — Final Validation

In [ ]:
mf = con.execute('SELECT * FROM model_features').fetchdf()

new_cols = [
    'on_loop_diuretic', 'on_ace_arb', 'on_beta_blocker',
    'on_aldosterone_antagonist', 'on_digoxin', 'on_anticoagulant',
    'num_unique_drugs', 'furosemide_max_dose', 'gdmt_score'
]

print('=== NEW MEDICATION FEATURES IN model_features ===')
print(f'Total shape: {mf.shape}')
print()
for col in new_cols:
    null_pct = mf[col].isna().mean() * 100
    if col in ['on_loop_diuretic','on_ace_arb','on_beta_blocker',
                'on_aldosterone_antagonist','on_digoxin','on_anticoagulant']:
        pos_pct = mf[col].mean() * 100
        print(f'  {col:35s}  null={null_pct:.0f}%  positive={pos_pct:.1f}%')
    else:
        print(f'  {col:35s}  null={null_pct:.0f}%  mean={mf[col].mean():.2f}')

print()
print('Readmission rate by GDMT score (clinical sanity check):')
gdmt_check = mf.groupby('gdmt_score')['readmitted_30d'].agg(['mean','count'])
gdmt_check.columns = ['readmit_rate', 'count']
gdmt_check['readmit_rate'] = (gdmt_check['readmit_rate'] * 100).round(1)
print(gdmt_check)
print()
print('(Expect: lower GDMT score → higher readmission rate)')

In [ ]:
print('=== NOTEBOOK 03b COMPLETE ===')
print()
print(f'model_features now has {mf.shape[1]} columns ({mf.shape[1]-9} original + 9 medication)')
print(f'Ready for: 04_Data_Preprocessing.ipynb')